# Tests

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np

from src.channels import AWGNChannel, BECChannel
from src.core import PolarCode, PolarEncoder, awgn_frozen_set, bec_frozen_set, sc_decode

N, K = 16, 8
possible_us: list[list[int]] = [[(i >> shift) & 1 for shift in (7, 6, 5, 4, 3, 2, 1, 0)] for i in range(64)]
right_counts: list[int] = []
verbose: bool = False

for i in range(1000):
    right_count: int = 0
    for u in possible_us:
        epsilon = 0.05
        frozen_bec = bec_frozen_set(N, K, epsilon)
        codeword_bec = PolarEncoder(PolarCode(N, K, frozen_bec)).encode(u)
        lrs_bec = BECChannel(epsilon).transmit(codeword_bec, mode='lrs')
        raw_estimate = sc_decode(frozen_bec, codeword_bec, lrs_bec, BECChannel(epsilon))
        clean_estimate = np.array([int(raw_estimate[i]) for i in range(raw_estimate.size) if i not in frozen_bec])
        if np.array_equal(u, clean_estimate):
            if verbose:
                print(f"Is {u} == {clean_estimate}?", True)
            right_count += 1
        else:
            if verbose:
                print(f"Is {u} == {clean_estimate}?", False)

    if verbose:
        print(f"{right_count} out of {len(possible_us)} were correctly decoded")
    right_counts.append(right_count)



In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

count_distribution = Counter(right_counts)

x = sorted(count_distribution.keys())
y = [count_distribution[count] for count in x]

plt.figure(figsize=(8, 4))
plt.bar(x, y, width=0.8)
plt.xlabel("Right count")
plt.ylabel("Number of experiments")
plt.title(f"Distribution of Right Counts out of {len(possible_us)}")
plt.xticks(x)
plt.grid(axis="y", alpha=0.3)
plt.show()

In [74]:
np.float16(1.0 / 0.0)

ZeroDivisionError: float division by zero